In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [2]:
sqlite_index.count()


83

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [4]:
## using RAG over a persistent index
from langchain_ollama import ChatOllama

from src.rag_helper import RAGBase

assistant = RAGBase(
    index=sqlite_index,
    llm_client=ChatOllama,
)

In [7]:
answer = assistant.rag("How do I get a certificate?")
print(answer)

To get a certificate, you need to pass the Capstone project.

Additionally:
*   You can only receive a certificate if you finish the course with a "live" cohort (the course does not award certificates for the self-paced mode).
*   If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


How it looks in a graoh:
```mermaid
flowchart TD

    subgraph ING["INGESTION"]
        direction LR
        FAQ[FAQ.json]
        INGESTOR[Ingestor<br/>parse, chunk, embed, metadata]
        FAQ --> INGESTOR
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    INGESTOR -->|Index Documents| DB
```

The ingestion process writes documents to the knowledge base.

The RAG assistant then reads from it:

```mermaid
flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
```

In [8]:
sqlite_index.close()